# __MODEL_LABEL_MARKDOWN__: model exploration

Run notebook 01 first. This notebook loads its saved dataset so you can try
feature treatments and fit models locally. When you have a configuration to
validate, export it to TOML and load it in notebook 03.


In [ ]:
from pathlib import Path

PROJECT_ROOT = next(
    candidate
    for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
    if (candidate / "pyproject.toml").is_file()
    and (candidate / "pricing_models").is_dir()
)

import pandas as pd
from superglm import (
    Categorical, Numeric, OrderedCategorical, Spline, SuperGLM, Tweedie, collapse_levels,
)

from pricing_pipeline.models.config import ValidationSplitConfig
from pricing_pipeline.notebook import (
    Clip, Log, Log1p, ModelRecipe, PricingDataset, PricingModelSpec, apply_transforms,
)

MODEL_DIR = PROJECT_ROOT / "pricing_models/__PACKAGE_NAME__"
DATASET_PATH = MODEL_DIR / ".local" / "dataset.joblib"


## Load the dataset from 01

Use the saved rows in their existing order. If you accept an enrichment or
source-data change here, put it in 01 and save the updated dataset before using 03.


In [ ]:
dataset = PricingDataset.load(DATASET_PATH)
df = dataset.df
display({"Rows": len(df), "Columns": len(df.columns)})
display(df.head())


## Set up the features

Define transforms and feature treatments here. A transform creates a model-input
column; its feature definition tells SuperGLM how to use it. Groupings, reference
levels and specials belong in these definitions too.

The commented examples show the syntax. Replace their column names and levels
with yours. `Log` requires positive values; `Log1p` computes log(1 + x).


In [ ]:
# Source-column transforms.
transforms = {
    # "log_feature": Log("positive_feature"),
    # "log1p_feature": Log1p("nonnegative_feature"),
    # "clipped_feature": Clip("__FEATURE_NAME__", lower=0, upper=100),
    # "log_exposure": Log("exposure"),
}
df = apply_transforms(dataset.df, transforms)

# Feature types, groupings and special levels.
features = {
    "__FEATURE_NAME__": Numeric(),
    "segment": Categorical(
        # base="A",
        # grouping=collapse_levels(df["segment"], groups={"BC": ["B", "C"]}),
    ),
    # "continuous_feature": Spline("cr", k=3, knot_strategy="quantile"),
    # "ordered_feature": OrderedCategorical(
    #     order=["1", "2", "3", "4", "5"],
    #     specials=["Unknown"],
    #     base="1",
    #     basis=Spline("cr", k=3, knot_strategy="quantile"),
    # ),
}


## Configure the model

Choose the target, family, weights and offset. The validation settings are saved
for 03; the fit below uses all the loaded rows.


In [ ]:
MODEL = PricingModelSpec(
    # Model.
    name="__MODEL_NAME__",
    label="__MODEL_LABEL__",
    model_type="__MODEL_TYPE__",
    deployment_slot="__DEPLOYMENT_SLOT__",

    # Data and features.
    dataset=dataset,
    target="__TARGET_NAME__",
    features=tuple(features),
    transforms=transforms,

    # Validation in notebook 03.
    validation=ValidationSplitConfig.kfold(
        n_splits=5, random_state=42, shuffle=True,
    ),

    # Optional offset and weights.
    # offset_column="log_exposure",
    # sample_weight_column="model_weight",
    # export_weight_column="rating_table_weight",
)

model = SuperGLM(
    family="poisson",  # For compound Tweedie: Tweedie(p=1.6).
    features=features,
    selection_penalty=0.0,
    retain_fit_state=False,
    discrete=True,
    n_bins=64,
)


## Fit locally

Edit the setup above and rerun from there to try another configuration.


In [ ]:
X = df.loc[:, list(MODEL.features)]
y = df[MODEL.target]
sample_weight = None if MODEL.sample_weight_column is None else df[MODEL.sample_weight_column]
offset = None if MODEL.offset_column is None else df[MODEL.offset_column]

model.fit_reml(X, y, sample_weight=sample_weight, offset=offset)


## Inspect the fit

These predictions use the fitting data. Use held-out data to assess performance;
03 runs the validation specified above. Add plots and comparisons as needed.


In [ ]:
predictions = model.predict(X, offset=offset)
results = pd.DataFrame({"actual": y, "prediction": predictions})
display(model.summary())
display(model.relativities(with_se=False, centering="native"))
display(results.head())


In [ ]:
# Add your plots, comparisons and alternative fits here.


## Save the configuration for 03

When you are ready, uncomment the cell below. It writes `prototype.toml` with
the feature definitions, groupings, specials, transforms and model settings.
The data and fitted coefficients stay separate.

In notebook 03, set `RECIPE_PATH = "prototype.toml"`. It loads these choices,
fits and validates against the saved dataset, then lets you save a model version.
To replace an existing config file, add `replace=True` to `save`.


In [ ]:
# recipe = ModelRecipe.from_model(model, spec=MODEL)
# recipe.save(MODEL_DIR / "prototype.toml")
